# Libraries

In [1]:
# 1. Base Framework: Always install PyTorch first to ensure proper CUDA bindings.
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 2. Core Framework: Install transformers before higher-level libraries.
!pip install -q "transformers==4.57.6" --break-system-packages

# 3. Compression Tools: Install tools that depend on the frameworks above.
!pip install -q compressed-tensors
!pip install -q -U llmcompressor

# 4. Evaluation Framework: Install lm-eval and its necessary extras.
!pip install -q "lm_eval[hf]"

# 5. Utilities: Standalone packages that don't deeply affect the ML framework dependencies.
!pip install -q langdetect --break-system-packages # for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 4.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 6.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
# --- Identity ---
user_name = "Abdelrahmanemam01"
email     = "abdulrahmanemam01@gmail.com"
repo      = "MahmoudOsama20/EdgeCompress"  # Fixed

# --- Model ---
QWEN       = False
model      = "Qwen3-4B"             if QWEN else "Llama-3.2-3B-Instruct"
MODEL_NAME       = "Llama-3.2-3B-Instruct-SparseGPT-AWQ"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit"
MODEL_PRETRAINED_FIX = MODEL_PRETRAINED.replace("/", "__")
SUB_FOLDER = "Sparsity/50"

# --- Reproducibility ---
SEED = 42

# --- Environment ---
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [8]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Evaluations

**Configurations**

In [9]:
# Task definition
TASKS = [
    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=14,  max_gen_toks=1024),

    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [ ]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "DEBUG",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)
    
    # Push To GitHub
    target_dir_in_repo = f"New_Evaluations/{model}/{MODEL_NAME}/{SUB_FOLDER}/results/{t.key}"
    source_folder = f"{results_dir}/{t.key}/{MODEL_PRETRAINED_FIX}"  # The Kaggle folder you want to upload
    remote_url = f"https://{token}@github.com/{repo}.git"
    
    # A temporary workspace in Kaggle to handle the cloning
    temp_repo_dir = "/kaggle/working/temp_git_repo" 
    # ---------------------

    # 2. Clean up any previous runs and clone the existing repository
    os.system(f"rm -rf {temp_repo_dir}")
    os.system(f"git clone {remote_url} {temp_repo_dir}")
    
    # 3. Create the target directory inside the cloned repo
    full_target_path = f"{temp_repo_dir}/{target_dir_in_repo}"
    os.makedirs(full_target_path, exist_ok=True)
    
    # 4. Copy the contents of your Kaggle folder into the target GitHub folder
    # (Using cp -r to recursively copy all files and subfolders)
    os.system(f"cp -r {source_folder}/* {full_target_path}/")
    
    # 5. Configure, commit, and push the changes
    os.system(f"""
        git -C {temp_repo_dir} config user.email "{email}" && \
        git -C {temp_repo_dir} config user.name "{user_name}" && \
        git -C {temp_repo_dir} add . && \
        git -C {temp_repo_dir} commit -m "Add Kaggle results to {target_dir_in_repo}" && \
        git -C {temp_repo_dir} push origin main
    """)
    
    print(f"Successfully pushed to {repo} inside the '{target_dir_in_repo}' folder!")
    
    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] gsm8k


2026-04-11:19:37:52 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-11:19:37:58 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-04-11:19:38:00 INFO     [evaluator:211] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-04-11:19:38:00 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-04-11:19:38:00 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit', 'subfolder': 'Sparsity/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'trust_remote_code': True}
2026-04-11:19:38:06 INFO     [models.huggingface:178] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
2026-04-11:

# Save reports

In [ ]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)